# Notebook 5 — Applications: RAG & Simple Tools (Quickstart)

A compact, runnable notebook demonstrating a minimal Retrieval-Augmented Generation (RAG) pattern using local embeddings (or a TF-IDF fallback) and a simple LLM call (transformers pipeline if available).

In [ ]:
# Cell 2: Minimal RAG-style retrieval + generation (CPU-friendly)
docs = [
    'CRISPR-Cas9 is a gene editing technology that allows precise DNA changes.',
    'DNA sequencing reads the order of nucleotides in genomes.',
    'Gene therapy uses genes to treat diseases.',
    'Protein folding determines function of proteins.'
]
# Simple embedding: use sentence-transformers if available, else TF-IDF
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    doc_emb = model.encode(docs)
    def embed(text):
        return model.encode([text])[0]
    print('Using sentence-transformers for embeddings')
except Exception:
    print('sentence-transformers not available, using TF-IDF fallback')
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer().fit(docs)
    doc_emb = vec.transform(docs).toarray()
    def embed(text):
        return vec.transform([text]).toarray()[0]

# Retrieve: cosine similarity
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, k=2):
    q = embed(query)
    sims = cosine_similarity([q], doc_emb)[0]
    idx = np.argsort(-sims)[:k]
    return [(docs[i], float(sims[i])) for i in idx]

query = 'How does CRISPR enable gene editing?'
print('Query:', query)
print('
Top retrieved docs:')
for d, s in retrieve(query):
    print(f'- {d} (score={s:.3f})')

# Optionally generate a short answer with Transformers (if available)
try:
    from transformers import pipeline
    gen = pipeline('text-generation', model='distilgpt2')
    context = ' '.join([r[0] for r in retrieve(query, k=2)])
    prompt = f'Based on the following context: {context}

Answer: {query}'
    out = gen(prompt, max_length=150, do_sample=False)
    print('
Generated answer:')
    print(out[0]['generated_text'])
except Exception as e:
    print('
Transformers generation not available; skip generation step.')

## Notes
- This notebook demonstrates the RAG pattern with minimal dependencies.
- Replace the fallback with FAISS/sentence-transformers for production.
- For privacy, keep documents local and avoid external APIs when working with sensitive biological data.